In [7]:
pip install twilio

Note: you may need to restart the kernel to use updated packages.


In [8]:
import logging
from twilio.rest import Client
from typing import Optional


class TwilioLineTypeInfo:
    def __init__(self, account_sid: str, auth_token: str):
        self.client = Client(account_sid, auth_token)
        self.log = logging.getLogger("Twilio")

    def get_line_type_info(self, phone_number: str) -> dict:
        """Fetch line_type_intelligence for a given phone number."""
        try:
            phone_number_instance = self.client.lookups.v2.phone_numbers(phone_number).fetch(
                fields="line_type_intelligence"
            )
        except Exception as e:
            # Just want to log the exception and move on (without re-raising)
            self.log.exception(
                f"[Twilio] failure while fetching line_type_intelligence for {phone_number}: {e}"
            )
            return {}

        return phone_number_instance.line_type_intelligence or {}

    def get_carrier_name(self, phone_number: str) -> Optional[str]:
        info = self.get_line_type_info(phone_number)
        return info.get("carrier_name")

    def get_line_type(self, phone_number: str) -> Optional[str]:
        info = self.get_line_type_info(phone_number)
        return info.get("type")


In [6]:
import os

# Try to get secrets from dbutils (Databricks), fall back to environment variables
try:
    ssid = dbutils.secrets.get(scope="local-dev", key="twillo-sid")
    token = dbutils.secrets.get(scope="local-dev", key="twillo-token")
except NameError:
    # Running locally, use environment variables
    ssid = os.getenv("TWILIO_ACCOUNT_SID", "")
    token = os.getenv("TWILIO_AUTH_TOKEN", "")

twilio_info = TwilioLineTypeInfo(ssid, token)

In [5]:
me =  twilio_info.get_line_type('+14085686874')
print(me)

ERROR:Twilio:[Twilio] failure while fetching line_type_intelligence for +14085686874: HTTP 401 error: Unable to fetch record: Authentication Error - No credentials provided
Traceback (most recent call last):
  File "/var/folders/7v/gm38h87d2xn4jrxydj2vp44m0000gn/T/ipykernel_4176/828521822.py", line 14, in get_line_type_info
    phone_number_instance = self.client.lookups.v2.phone_numbers(phone_number).fetch(
                            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/apple/sandbox/sound-of-e2/.venv/lib/python3.12/site-packages/twilio/rest/lookups/v2/phone_number.py", line 308, in fetch
    payload = self._version.fetch(
              ^^^^^^^^^^^^^^^^^^^^
  File "/Users/apple/sandbox/sound-of-e2/.venv/lib/python3.12/site-packages/twilio/base/version.py", line 141, in fetch
    return self._parse_fetch(method, uri, response)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/apple/sandbox/sound-of-e2/.venv/lib/python3.12/site-packa

None
